In [ ]:
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# Connect
client = chromadb.HttpClient(
    host=os.getenv("CHROMA_HOST", "localhost"),
    port=int(os.getenv("CHROMA_PORT", "8000")),
    settings=Settings(anonymized_telemetry=False)
)

# Load collection
collection = client.get_collection(name="jobs_search_master_vector02")

# Embedder ต้องตรงกับตอนสร้าง
embedder = SentenceTransformer("intfloat/multilingual-e5-large")

queries = [
    "หางาน Software Engineer / Develope",
    "หางานในด้านการออกแบบ (Design)",
    "หางานในด้าน IT Infrastructure / Server / Network",
]

for i, query in enumerate(queries, 1):
    print(f"\n{'='*80}")
    print(f"🔍 Query {i}: {query}")
    print(f"{'='*80}")

    query_emb = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_emb,
        n_results=10,
        include=["documents", "metadatas", "distances"]
    )

    if results["documents"] and results["documents"][0]:
        for j, (doc, meta, dist) in enumerate(
            zip(results["documents"][0],
                results["metadatas"][0],
                results["distances"][0]), 1
        ):
            print(f"\n📋 อันดับ {j} (distance: {dist:.4f})")
            print(f"   🔹 id: {meta.get('id', 'N/A')}")
            print(f"   jobpost_id: {meta.get('jobpost_id', 'N/A')}")
            print(f"   ตำแหน่ง: {meta.get('position', 'N/A')}")
            print(f"   บริษัท: {meta.get('company_name', 'N/A')}")
            print(f"   จังหวัด: {meta.get('job_province', 'N/A')}")
            print(f"   เงินเดือน: {meta.get('salary_start', '')} - {meta.get('salary_end', '')}")
            print(f"   รายละเอียด: {doc[:150]}..." if len(doc) > 150 else f"   รายละเอียด: {doc}")
    else:
        print("   ❌ ไม่พบผลลัพธ์")
